# **Understanding Dataset**

Import libraries

In [106]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, silhouette_score, pairwise_distances_argmin_min
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
import matplotlib.pyplot as plt
import matplotlib
from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from matplotlib.colors import Normalize
from matplotlib.lines import Line2D

import ast
from scipy.spatial.distance import cdist

import matplotlib.pyplot as plt
import seaborn as sns

# **Understanding Dataset**

Read dataframe

In [107]:
fish = pd.read_csv('Phishing_Legitimate_full.csv')

fish.shape
fish.columns


Index(['id', 'NumDots', 'SubdomainLevel', 'PathLevel', 'UrlLength', 'NumDash',
       'NumDashInHostname', 'AtSymbol', 'TildeSymbol', 'NumUnderscore',
       'NumPercent', 'NumQueryComponents', 'NumAmpersand', 'NumHash',
       'NumNumericChars', 'NoHttps', 'RandomString', 'IpAddress',
       'DomainInSubdomains', 'DomainInPaths', 'HttpsInHostname',
       'HostnameLength', 'PathLength', 'QueryLength', 'DoubleSlashInPath',
       'NumSensitiveWords', 'EmbeddedBrandName', 'PctExtHyperlinks',
       'PctExtResourceUrls', 'ExtFavicon', 'InsecureForms',
       'RelativeFormAction', 'ExtFormAction', 'AbnormalFormAction',
       'PctNullSelfRedirectHyperlinks', 'FrequentDomainNameMismatch',
       'FakeLinkInStatusBar', 'RightClickDisabled', 'PopUpWindow',
       'SubmitInfoToEmail', 'IframeOrFrame', 'MissingTitle',
       'ImagesOnlyInForm', 'SubdomainLevelRT', 'UrlLengthRT',
       'PctExtResourceUrlsRT', 'AbnormalExtFormActionR', 'ExtMetaScriptLinkRT',
       'PctExtNullSelfRedirectHyperl

As you can see there are 50 features and 10000 entries, notice the  Class_LABEL feature will be 1 if it is not scam and 0 otherwise. We will use supervised learning to fully utilize this feature.


In [108]:
fish.head()

,id,NumDots,SubdomainLevel,PathLevel,UrlLength,NumDash,NumDashInHostname,AtSymbol,TildeSymbol,NumUnderscore,...,IframeOrFrame,MissingTitle,ImagesOnlyInForm,SubdomainLevelRT,UrlLengthRT,PctExtResourceUrlsRT,AbnormalExtFormActionR,ExtMetaScriptLinkRT,PctExtNullSelfRedirectHyperlinksRT,CLASS_LABEL
0,1,3,1,5,72,0,0,0,0,0,...,0,0,1,1,0,1,1,-1,1,1
1,2,3,1,3,144,0,0,0,0,2,...,0,0,0,1,-1,1,1,1,1,1
2,3,3,1,2,58,0,0,0,0,0,...,0,0,0,1,0,-1,1,-1,0,1
3,4,3,1,6,79,1,0,0,0,0,...,0,0,0,1,-1,1,1,1,-1,1
4,5,3,0,4,46,0,0,0,0,0,...,1,0,0,1,1,-1,0,-1,-1,1


# **Preprocessing data**

Lets see how many missing entries are located in each feature

In [109]:
missing_data = fish.isna().sum().to_dict()
print("MISSING DATA IN COL:", missing_data)

MISSING DATA IN COL: {'id': 0, 'NumDots': 0, 'SubdomainLevel': 0, 'PathLevel': 0, 'UrlLength': 0, 'NumDash': 0, 'NumDashInHostname': 0, 'AtSymbol': 0, 'TildeSymbol': 0, 'NumUnderscore': 0, 'NumPercent': 0, 'NumQueryComponents': 0, 'NumAmpersand': 0, 'NumHash': 0, 'NumNumericChars': 0, 'NoHttps': 0, 'RandomString': 0, 'IpAddress': 0, 'DomainInSubdomains': 0, 'DomainInPaths': 0, 'HttpsInHostname': 0, 'HostnameLength': 0, 'PathLength': 0, 'QueryLength': 0, 'DoubleSlashInPath': 0, 'NumSensitiveWords': 0, 'EmbeddedBrandName': 0, 'PctExtHyperlinks': 0, 'PctExtResourceUrls': 0, 'ExtFavicon': 0, 'InsecureForms': 0, 'RelativeFormAction': 0, 'ExtFormAction': 0, 'AbnormalFormAction': 0, 'PctNullSelfRedirectHyperlinks': 0, 'FrequentDomainNameMismatch': 0, 'FakeLinkInStatusBar': 0, 'RightClickDisabled': 0, 'PopUpWindow': 0, 'SubmitInfoToEmail': 0, 'IframeOrFrame': 0, 'MissingTitle': 0, 'ImagesOnlyInForm': 0, 'SubdomainLevelRT': 0, 'UrlLengthRT': 0, 'PctExtResourceUrlsRT': 0, 'AbnormalExtFormActionR

ight none


In [110]:
# Scaling and Normalization
from sklearn.preprocessing import StandardScaler

# Drop the 'id' column and target variable
X = fish.drop(columns=['CLASS_LABEL', 'id'])  # Features
y = fish['CLASS_LABEL']  # Target

# Scale numerical features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Convert scaled data back to a DataFrame
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)


In [111]:
# Feature Selection/Reduction
# Calculate correlation with the target variable
correlation = fish.corr()['CLASS_LABEL'].sort_values(ascending=False)
print("Feature Correlation with CLASS_LABEL:\n", correlation)

# Set a threshold for correlation (e.g., > 0.1 or < -0.1)
relevant_features = correlation[(correlation > 0.1) | (correlation < -0.1)].index
print("Relevant Features Before Filtering:", relevant_features)

# Filter relevant features to include only those in X_scaled
relevant_features = [feature for feature in relevant_features if feature in X_scaled.columns]
print("Relevant Features After Filtering:", relevant_features)

# Filter dataset for relevant features
X_selected = X_scaled[relevant_features]

# Apply PCA to retain 95% of the variance
pca = PCA(n_components=0.95)  # Retain 95% variance
X_pca = pca.fit_transform(X_selected)

print("Explained Variance Ratio:", pca.explained_variance_ratio_)
print("Number of Components:", pca.n_components_)


Feature Correlation with CLASS_LABEL:
 CLASS_LABEL                           1.000000
FrequentDomainNameMismatch            0.463956
PctNullSelfRedirectHyperlinks         0.342806
InsecureForms                         0.316380
NumDots                               0.294111
PctExtHyperlinks                      0.259728
NumSensitiveWords                     0.255208
PathLevel                             0.229450
AbnormalExtFormActionR                0.185799
UrlLengthRT                           0.169513
HostnameLength                        0.169157
NumDashInHostname                     0.150444
EmbeddedBrandName                     0.141790
IpAddress                             0.132291
MissingTitle                          0.116693
ExtMetaScriptLinkRT                   0.111150
DomainInSubdomains                    0.100452
TildeSymbol                           0.095864
RightClickDisabled                    0.074900
ExtFavicon                            0.069140
PctExtResourceUrlsRT 

In [112]:
# Outlier Detection and Removal
# Calculate Z-scores for each feature
z_scores = np.abs((X_selected - X_selected.mean()) / X_selected.std())

# Set threshold for Z-scores (adjust as needed)
threshold = 4  # Allow more variability
non_outliers = (z_scores < threshold).all(axis=1)

# Filter the dataset
X_cleaned = X_selected[non_outliers]
y_cleaned = y[non_outliers]

print("Original Data Shape:", X_selected.shape)
print("Data Shape After Outlier Removal:", X_cleaned.shape)

Original Data Shape: (10000, 24)
Data Shape After Outlier Removal: (7915, 24)


In [113]:
# Cleaned Dataset
# Combine cleaned features and target
preprocessed_data = pd.DataFrame(X_cleaned, columns=X_selected.columns)
preprocessed_data['CLASS_LABEL'] = y_cleaned.values

# Save preprocessed data to a CSV file
preprocessed_data.to_csv('preprocessed_phishing_data.csv', index=False)

new_data = pd.read_csv('preprocessed_phishing_data.csv')
print("Preprocessing complete. Data saved to 'preprocessed_phishing_data.csv'.")


Preprocessing complete. Data saved to 'preprocessed_phishing_data.csv'.


In [114]:
print("Shape of the cleaned dataset:", new_data.shape)

print("Columns in the cleaned dataset:")
print(new_data.columns.tolist())

print("Summary statistics of the cleaned dataset:")
print(new_data.describe())

print("Class distribution in the cleaned dataset:")
print(new_data['CLASS_LABEL'].value_counts())

print("First few rows of the cleaned dataset:")
print(new_data.head())


Shape of the cleaned dataset: (7915, 25)
Columns in the cleaned dataset:
['FrequentDomainNameMismatch', 'PctNullSelfRedirectHyperlinks', 'InsecureForms', 'NumDots', 'PctExtHyperlinks', 'NumSensitiveWords', 'PathLevel', 'AbnormalExtFormActionR', 'UrlLengthRT', 'HostnameLength', 'NumDashInHostname', 'EmbeddedBrandName', 'IpAddress', 'MissingTitle', 'ExtMetaScriptLinkRT', 'DomainInSubdomains', 'ExtFormAction', 'DomainInPaths', 'AbnormalFormAction', 'NumQueryComponents', 'IframeOrFrame', 'SubmitInfoToEmail', 'NumDash', 'PctExtNullSelfRedirectHyperlinksRT', 'CLASS_LABEL']
Summary statistics of the cleaned dataset:
       FrequentDomainNameMismatch  PctNullSelfRedirectHyperlinks  \
count                 7915.000000                    7915.000000   
mean                    -0.112532                       0.029660   
std                      0.911895                       1.032588   
min                     -0.523806                      -0.435776   
25%                     -0.523806          